<a href="https://colab.research.google.com/github/tusharchouhan/banking-compliance-llm-assignment-1a/blob/master/assignment_part_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1A — Part A
Run the PDF, cleaning, split, token packing, CPT, perplexity, and forgetting stages.

In [17]:
# GPU Information
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [18]:
#6. Clone your GitHub repository
!git clone https://github.com/tusharchouhan/banking-compliance-llm-assignment-1a.git

Cloning into 'banking-compliance-llm-assignment-1a'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 63 (delta 11), reused 47 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 4.90 MiB | 11.16 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [19]:
%cd /content/banking-compliance-llm-assignment-1a

/content/banking-compliance-llm-assignment-1a


In [20]:
#Verify:
from pathlib import Path

PROJECT = Path.cwd()
print(PROJECT)
print((PROJECT / "src").exists())
print((PROJECT / "run_pipeline.py").exists())

/content/banking-compliance-llm-assignment-1a
True
True


In [21]:
#7. Install dependencies
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 21.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.4 MB/s eta 0:00:00


In [22]:
# Verify the important packages:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch: 2.11.0+cpu
CUDA available: False
GPU: No GPU


In [23]:
#Mount Google Drive and copy the PDFs
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
#Then copy the PDF directory into the cloned repository:
from pathlib import Path
import shutil

PROJECT = Path.cwd()
DRIVE_RAW = Path("/content/drive/MyDrive/LLM_ASSIGNMENT_DATA/raw_pdfs")
LOCAL_RAW = PROJECT / "data" / "raw_pdfs"

if not DRIVE_RAW.exists():
    raise FileNotFoundError(f"Drive folder not found: {DRIVE_RAW}")

shutil.copytree(DRIVE_RAW, LOCAL_RAW, dirs_exist_ok=True)

pdfs = sorted(LOCAL_RAW.rglob("*.pdf"))
total_mb = sum(pdf.stat().st_size for pdf in pdfs) / (1024 * 1024)

print("PDF count:", len(pdfs))
print("Total size:", round(total_mb, 2), "MB")
print("Example:", pdfs[0] if pdfs else "No PDFs found")

PDF count: 127
Total size: 159.65 MB
Example: /content/banking-compliance-llm-assignment-1a/data/raw_pdfs/Basel/201712-standards-basel-iii-finalising-post-crisis-reforms.pdf


In [25]:
from pathlib import Path
import sys
ROOT=Path.cwd() / 'LLM_ASSIGNMENT' if (Path.cwd()/'LLM_ASSIGNMENT').exists() else Path.cwd()
sys.path.insert(0,str(ROOT))

In [26]:
!python -m src.data.pdf_extractor

2026-08-31 11:46:04,519 | INFO | pdf_extractor | Extracted 201712-standards-basel-iii-finalising-post-crisis-reforms.pdf: 162 pages, 535392 characters
2026-08-31 11:46:04,551 | INFO | pdf_extractor | Extracted Addendum - Guidelines for Corporate Governance for Insurers in India.pdf: 1 pages, 1971 characters
2026-08-31 11:46:05,048 | INFO | pdf_extractor | Extracted GUIDELINES FOR INTENSIVE EXAMINATION OF PUBLIC PROCUREMENT CONTRACTS BY.pdf: 10 pages, 17112 characters
2026-08-31 11:46:08,306 | INFO | pdf_extractor | Extracted Guidelines for Corporate Governance for insurers in India.pdf: 67 pages, 134077 characters
2026-08-31 11:46:08,781 | INFO | pdf_extractor | Extracted Guidelines on Appointment of Insurance Agents 2015 –Instructions to Insurer.pdf: 3 pages, 5536 characters
2026-08-31 11:46:11,529 | INFO | pdf_extractor | Extracted Guidelines on Cross Border Re-insurers.pdf: 15 pages, 14814 characters
2026-08-31 11:46:11,558 | INFO | pdf_extractor | Extracted Guidelines on Establishm

In [27]:
import pandas as pd

pd.read_csv("reports/tables/extraction_report.csv").head(20)

,source_pdf,output_txt,pages,characters,status
0,Basel/201712-standards-basel-iii-finalising-po...,Basel/201712-standards-basel-iii-finalising-po...,162,535392,ok
1,IRDA/Addendum - Guidelines for Corporate Gover...,IRDA/Addendum - Guidelines for Corporate Gover...,1,1971,ok
2,IRDA/GUIDELINES FOR INTENSIVE EXAMINATION OF P...,IRDA/GUIDELINES FOR INTENSIVE EXAMINATION OF P...,10,17112,ok
3,IRDA/Guidelines for Corporate Governance for i...,IRDA/Guidelines for Corporate Governance for i...,67,134077,ok
4,IRDA/Guidelines on Appointment of Insurance Ag...,IRDA/Guidelines on Appointment of Insurance Ag...,3,5536,ok
5,IRDA/Guidelines on Cross Border Re-insurers.pdf,IRDA/Guidelines on Cross Border Re-insurers.txt,15,14814,ok
6,IRDA/Guidelines on Establishment and Closure o...,IRDA/Guidelines on Establishment and Closure o...,16,181,ok
7,IRDA/Guidelines on Indian Owned and Controlled...,IRDA/Guidelines on Indian Owned and Controlled...,2,3103,ok
8,IRDA/Guidelines on Indian owned and controlled...,IRDA/Guidelines on Indian owned and controlled...,5,7980,ok
9,IRDA/Guidelines on Indian owned and controlled...,IRDA/Guidelines on Indian owned and controlled...,5,7980,ok


In [28]:
!python -m src.data.clean_corpus

2026-08-31 11:51:50,123 | INFO | clean_corpus | Cleaning complete: 127 -> 118 documents


In [29]:
pd.read_csv("reports/tables/cleaning_report.csv")

,step,documents,removed,impact
0,input,127,0,baseline
1,input,127,0,NaN
2,length_filter,125,2,NaN
3,repetition_filter,125,0,NaN
4,deduplication,123,2,NaN
5,english_filter,118,5,greatest


In [30]:
#Train/evaluation split
!python -m src.data.train_eval_split

2026-08-31 11:51:50,539 | INFO | train_eval_split | Split 118 documents into 106 train and 12 eval files


In [31]:
pd.read_csv("reports/tables/split_report.csv")

,split,documents,fraction,seed
0,train,106,90% target,42
1,eval,12,10% target,42


In [32]:
#Tokenization and packing
!python -m src.data.tokenize_and_pack --seq-len 1024

2026-08-31 11:51:59,574 | INFO | numexpr.utils | NumExpr defaulting to 2 threads.
2026-08-31 11:52:00,548 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/HuggingFaceTB/SmolLM2-360M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-31 11:52:00,549 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-31 11:52:00,560 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/HuggingFaceTB/SmolLM2-360M/f8027fd0eaeea54caa13c31d31b9fdc459c38b49/config.json "HTTP/1.1 200 OK"
2026-08-31 11:52:00,907 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/HuggingFaceTB/SmolLM2-360M/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-31 11:52:00,918 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/HuggingFaceTB/SmolLM2-360M/f8027fd0eaeea54caa13c31d3

In [33]:
pd.read_csv("reports/tables/token_statistics.csv")

,model_id,sequence_length,documents,total_tokens_with_bos_eos,average_document_tokens,packed_sequences,discarded_tail_tokens,seed
0,HuggingFaceTB/SmolLM2-360M,1024,106,2868001,27054.61,2800,801,42


In [34]:
# 10. Run model inspection and baseline inference
!python -m src.evaluation.model_inspection
!python -m src.evaluation.baseline_inference

2026-08-31 11:52:33,858 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/HuggingFaceTB/SmolLM2-360M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-31 11:52:33,858 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-31 11:52:33,870 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/HuggingFaceTB/SmolLM2-360M/f8027fd0eaeea54caa13c31d31b9fdc459c38b49/config.json "HTTP/1.1 200 OK"
2026-08-31 11:52:36,507 | INFO | numexpr.utils | NumExpr defaulting to 2 threads.
2026-08-31 11:52:36,868 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/HuggingFaceTB/SmolLM2-360M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-31 11:52:36,880 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/HuggingFaceTB/SmolLM2-360M/f8027fd0eaeea54caa13c31d31b9fdc459c

In [35]:
#Display Report
pd.read_csv("reports/tables/model_architecture.csv")

,model_id,total_parameters,trainable_parameters,decoder_layers,attention_heads,hidden_size,vocabulary_size,head_dimension,lm_head_output_dimension,lm_head_matches_vocab
0,HuggingFaceTB/SmolLM2-360M,361821120,361821120,32,15,960,49152,64,49152,True


In [36]:
pd.read_csv("results/baseline_outputs/baseline_outputs.csv")

,prompt,response
0,What is KYC?,KYC stands for Know Your Customer. It is a pro...
1,What is LTV Ratio?,LTV ratio is a ratio that measures the percent...
2,What is Basel III?,Basel III is a new set of rules that banks mus...


In [37]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [38]:
!python -m src.training.cpt_train --max-steps 500

2026-08-31 11:54:08,314 | INFO | datasets | TensorFlow version 2.20.0 available.
2026-08-31 11:54:08,315 | INFO | datasets | JAX version 0.11.1 available.
2026-08-31 11:54:09,154 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/HuggingFaceTB/SmolLM2-360M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-31 11:54:09,155 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-31 11:54:09,165 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/HuggingFaceTB/SmolLM2-360M/f8027fd0eaeea54caa13c31d31b9fdc459c38b49/config.json "HTTP/1.1 200 OK"
2026-08-31 11:54:09,259 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/HuggingFaceTB/SmolLM2-360M/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-31 11:54:09,269 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/

In [39]:
!python -m src.evaluation.loss_analysis

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/banking-compliance-llm-assignment-1a/src/evaluation/loss_analysis.py", line 37, in <module>
    a=p.parse_args();run(a.input,a.csv,a.figure,a.summary)
                     ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/banking-compliance-llm-assignment-1a/src/evaluation/loss_analysis.py", line 15, in run
    raw=json.loads(input_path.read_text(encoding="utf-8"))
                   ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/pathlib/_local.py", line 546, in read_text
    return PathBase.read_text(self, encoding, errors, newline)
           ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/pathlib/_abc.py", line 632, in read_text
    with self.open(mode='r', encoding=encoding, errors=errors, newline=newline) as f:
         ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [40]:
from IPython.display import Image, display

display(Image(filename="reports/figures/loss_curve.png"))

FileNotFoundError: [Errno 2] No such file or directory: 'reports/figures/loss_curve.png'

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content/banking-compliance-llm-assignment-1a

In [ ]:
from pathlib import Path
import shutil

PROJECT = Path("/content/banking-compliance-llm-assignment-1a")
SAVE_DIR = Path("/content/drive/MyDrive/LLM_ASSIGNMENT_SAVED_RUN")

items = [
    "data/eval_corpus",
    "data/train_corpus",
    "data/train_packed.parquet",
    "reports",
    "results",
    "models/cpt_model",
]

for item in items:
    source = PROJECT / item
    destination = SAVE_DIR / item

    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    elif source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

print("Saved completed work to:", SAVE_DIR)

In [ ]:
# 12. Run perplexity and forgetting evaluation
!python -m src.evaluation.perplexity


In [ ]:
!python -m src.evaluation.forgetting_check

In [ ]:
pd.read_csv("results/perplexity/ppl_results.csv")

In [ ]:
from IPython.display import Markdown, display

display(Markdown(
    Path("results/forgetting_check/forgetting_comparison.md").read_text()
))

In [ ]:
#13. Create the instruction dataset
!python -m src.data.create_instruction_dataset --minimum 100

In [ ]:
#Display the split report:
import json

print(json.dumps(
    json.loads(Path("data/instruction_dataset/split_report.json").read_text()),
    indent=2
))

In [ ]:
#Verify the number of records:
!wc -l data/instruction_dataset/instruction_dataset.jsonl
!wc -l data/instruction_dataset/train.jsonl
!wc -l data/instruction_dataset/eval.jsonl

In [ ]:
#14. Train the three QLoRA adapters
!python -m src.qlora.train_adapter_A --max-steps 500


In [ ]:
!python -m src.qlora.train_adapter_B --max-steps 500

In [ ]:
!python -m src.qlora.train_adapter_C --max-steps 500

In [ ]:
#15. Compare the adapters
!python -m src.evaluation.adapter_comparison
!python -m src.evaluation.build_report

In [ ]:
#Display:
pd.read_csv("results/adapter_comparison/adapter_comparison.csv")

In [ ]:
16. Save everything to Google Drive
from pathlib import Path
import shutil

PROJECT = Path.cwd()
RUN_DIR = Path("/content/drive/MyDrive/LLM_ASSIGNMENT_RESULTS")

for folder in ["data", "models", "reports", "results"]:
    shutil.copytree(
        PROJECT / folder,
        RUN_DIR / folder,
        dirs_exist_ok=True
    )

print("Saved experiment outputs to:", RUN_DIR)